# Etapa 4.1 - Persistencia con Redis

Dominio: comercio minorista de suplementos deportivos. Usamos Redis como capa de acceso rapido (estructuras en memoria, acceso por clave en O(1)) para los datos que se consultan por ID y representan el estado actual del sistema.

## 1. Configuracion del entorno

Redis local con Docker:

```bash
docker run -d -p 6379:6379 --name redis-local-ibd redis:7.2-alpine
pip install redis
```

Conexion a `localhost:6379` con `decode_responses=True`.

In [1]:
# !pip install redis
import redis

r = redis.Redis(host="localhost", port=6379, decode_responses=True)
print("PING:", r.ping())
print("Redis version:", r.info("server")["redis_version"])

PING: True
Redis version: 7.2.14


### Limpieza idempotente
Borramos solo las claves de este ejercicio (por prefijo, con `SCAN`) para poder re-ejecutar la notebook.

In [2]:
PREFIJOS = ["stock:producto:", "stock:sucursal:", "producto:",
            "ventas:hoy:", "caja:", "pedidos:",
            "carrito:", "reserva:", "sesion:"]

borradas = 0
for pref in PREFIJOS:
    for k in r.scan_iter(match=pref + "*"):
        r.delete(k)
        borradas += 1
print(f"Claves previas del ejercicio borradas: {borradas}")

Claves previas del ejercicio borradas: 27


## 2. Datos del dominio modelados (4.1.1)

Modelamos tres datos: stock por producto/sucursal (hashes espejo), perfil de producto (hash) y contador de ventas del dia (string). Primero definimos un catalogo de ejemplo.

In [3]:
import random
random.seed(42)

PRODUCTOS = [
    {"id": 1,  "nombre": "Whey Protein 100%",      "precio": 29000, "marca": "Ena Sport"},
    {"id": 2,  "nombre": "Isolate Whey Protein",   "precio": 42000, "marca": "Dymatize"},
    {"id": 3,  "nombre": "Creatina Monohidrato",   "precio": 22000, "marca": "Star Nutrition"},
    {"id": 4,  "nombre": "BCAA 2:1:1",             "precio": 15000, "marca": "Nutrilab"},
    {"id": 5,  "nombre": "Glutamina Micronizada",  "precio": 13500, "marca": "Pulver"},
    {"id": 6,  "nombre": "Quemador Termogenico",   "precio": 12500, "marca": "MuscleTech"},
    {"id": 7,  "nombre": "Pre-Entreno C4",         "precio": 33000, "marca": "BSN"},
    {"id": 8,  "nombre": "Multivitaminico Diario", "precio": 8000,  "marca": "Universal"},
    {"id": 9,  "nombre": "Omega 3 Ultra",          "precio": 9900,  "marca": "Optimum"},
    {"id": 10, "nombre": "Colageno Hidrolizado",   "precio": 18500, "marca": "Hoch Sport"},
]

SUCURSALES = {1: "Palermo", 2: "Belgrano", 3: "Centro", 4: "Caballito"}

print("Productos:", len(PRODUCTOS), "| Sucursales:", len(SUCURSALES))

Productos: 10 | Sucursales: 4


## 3. Stock bidireccional

Mantenemos dos hashes espejo: `stock:producto:<id>` (campo = sucursal) y `stock:sucursal:<id>` (campo = producto). Los cargamos con un pipeline.

In [4]:
# Stock inicial (producto, sucursal) -> cantidad
stock = {}
for p in PRODUCTOS:
    for sid in SUCURSALES:
        stock[(p["id"], sid)] = random.randint(20, 300)

# Lo escribimos en los dos hashes espejo con un pipeline
pipe = r.pipeline()
for (pid, sid), cant in stock.items():
    pipe.hset(f"stock:producto:{pid}", sid, cant)
    pipe.hset(f"stock:sucursal:{sid}", pid, cant)
pipe.execute()

print("Hashes de stock cargados.")
print("Ejemplo - stock:producto:1 =", r.hgetall("stock:producto:1"))

Hashes de stock cargados.
Ejemplo - stock:producto:1 = {'1': '77', '2': '32', '3': '160', '4': '145'}


### Consultas
Stock de un producto en una sucursal (`HGET`), en todas (`HGETALL`) y todos los productos de una sucursal.

In [5]:
pid, sid = 1, 3
cant = r.hget(f"stock:producto:{pid}", sid)
print(f"O(1)  Stock del producto {pid} en {SUCURSALES[sid]}: {cant} unidades")

print(f"\nStock del producto {pid} en todas las sucursales:")
for s, c in r.hgetall(f"stock:producto:{pid}").items():
    print(f"   {SUCURSALES[int(s)]:<12} {c:>4}")

nombre = {p["id"]: p["nombre"] for p in PRODUCTOS}
print(f"\nInventario de la sucursal {SUCURSALES[sid]}:")
for pr, c in r.hgetall(f"stock:sucursal:{sid}").items():
    print(f"   [{pr:>2}] {nombre[int(pr)]:<24} {c:>4}")

O(1)  Stock del producto 1 en Centro: 160 unidades

Stock del producto 1 en todas las sucursales:
   Palermo        77
   Belgrano       32
   Centro        160
   Caballito     145

Inventario de la sucursal Centro:
   [ 1] Whey Protein 100%         160
   [ 2] Isolate Whey Protein       72
   [ 3] Creatina Monohidrato       36
   [ 4] BCAA 2:1:1                139
   [ 5] Glutamina Micronizada     299
   [ 6] Quemador Termogenico      162
   [ 7] Pre-Entreno C4            194
   [ 8] Multivitaminico Diario    192
   [ 9] Omega 3 Ultra              69
   [10] Colageno Hidrolizado       42


### Actualizacion + verificacion
Una venta descuenta stock con `HINCRBY` sobre los dos hashes dentro de una transaccion; luego releemos ambas vistas para comprobar que coinciden.

In [6]:
pid, sid, vendidas = 1, 3, 2
antes = int(r.hget(f"stock:producto:{pid}", sid))

# Descontamos en ambos hashes dentro de una transaccion
tx = r.pipeline(transaction=True)
tx.hincrby(f"stock:producto:{pid}", sid, -vendidas)
tx.hincrby(f"stock:sucursal:{sid}", pid, -vendidas)
tx.execute()

desde_producto = int(r.hget(f"stock:producto:{pid}", sid))
desde_sucursal = int(r.hget(f"stock:sucursal:{sid}", pid))

print(f"Stock antes de la venta : {antes}")
print(f"Vendidas               : {vendidas}")
print(f"Stock (vista producto) : {desde_producto}")
print(f"Stock (vista sucursal) : {desde_sucursal}")
print("Consistencia entre las dos vistas:",
      "OK" if desde_producto == desde_sucursal == antes - vendidas else "DESFASADO")

Stock antes de la venta : 160
Vendidas               : 2
Stock (vista producto) : 158
Stock (vista sucursal) : 158
Consistencia entre las dos vistas: OK


## 4. Perfil de producto (hash)
Datos de cada producto leidos por ID. `HGET` trae un campo; `HGETALL`, el documento completo.

In [7]:
pipe = r.pipeline()
for p in PRODUCTOS:
    pipe.hset(f"producto:{p['id']}",
              mapping={"nombre": p["nombre"], "precio": p["precio"], "marca": p["marca"]})
pipe.execute()

print("Precio del producto 7 (HGET):", r.hget("producto:7", "precio"))
print("Perfil completo del producto 7 (HGETALL):", r.hgetall("producto:7"))

Precio del producto 7 (HGET): 33000
Perfil completo del producto 7 (HGETALL): {'nombre': 'Pre-Entreno C4', 'precio': '33000', 'marca': 'BSN'}


### Actualizacion + verificacion
Cambio de precio del producto 7 con `HSET`.

In [8]:
print("Precio antes:", r.hget("producto:7", "precio"))
r.hset("producto:7", "precio", 35000)
print("Precio despues:", r.hget("producto:7", "precio"))

Precio antes: 33000
Precio despues: 35000


## 5. Contador de ventas del dia (string)
Contador por sucursal con `INCR` (atomico) y total facturado con `INCRBYFLOAT`.

In [9]:
sid = 3
r.set(f"ventas:hoy:sucursal:{sid}", 0)
r.set(f"caja:sucursal:{sid}", 0)

# 5 ventas: cada una suma al contador y a la caja del dia
for _ in range(5):
    r.incr(f"ventas:hoy:sucursal:{sid}")
    r.incrbyfloat(f"caja:sucursal:{sid}", 29000.00)

print(f"Ventas de hoy en {SUCURSALES[sid]}:", r.get(f"ventas:hoy:sucursal:{sid}"))
print(f"Facturado de hoy en {SUCURSALES[sid]}: $", r.get(f"caja:sucursal:{sid}"))

Ventas de hoy en Centro: 5
Facturado de hoy en Centro: $ 145000


## Nuevo requerimiento: ventas online con envio
El negocio incorpora venta online con envio: el pedido no se entrega en el acto, queda pendiente de despacho. Modelamos esa cola de pendientes con una lista de Redis.

## 7. Lista como cola de pedidos (4.1.2)
Cola FIFO con una lista: `RPUSH` encola al final (nueva venta) y `LPOP` saca del frente (despacho). Cada pedido se serializa como JSON. Clave: `pedidos:pendientes`.

In [10]:
import json
COLA = "pedidos:pendientes"
r.delete(COLA)
random.seed(7)

def nuevo_pedido(venta_id):
    sid = random.choice(list(SUCURSALES))
    items = random.sample(PRODUCTOS, random.randint(1, 4))
    return {
        "venta_id": venta_id,
        "sucursal": SUCURSALES[sid],
        "cliente_id": random.randint(1, 1500),
        "n_items": len(items),
        "total": sum(p["precio"] for p in items),
        "hora": f"{random.randint(9, 20):02d}:{random.randint(0, 59):02d}",
    }

def encolar_pedido(pedido):
    r.rpush(COLA, json.dumps(pedido))
    return pedido

def despachar_pedido():
    raw = r.lpop(COLA)
    return json.loads(raw) if raw else None

for vid in range(1050, 1055):
    p = encolar_pedido(nuevo_pedido(vid))
    print(f"Encolada venta {p['venta_id']} ({p['sucursal']}, ${p['total']})")

print("\nPedidos pendientes (LLEN):", r.llen(COLA))

Encolada venta 1050 (Centro, $62000)
Encolada venta 1051 (Centro, $9900)
Encolada venta 1052 (Caballito, $89000)
Encolada venta 1053 (Palermo, $47500)
Encolada venta 1054 (Palermo, $38900)

Pedidos pendientes (LLEN): 5


### Consultas sobre la cola
`LRANGE` (contenido), `LLEN` (tamano) y `LINDEX 0` (proximo a despachar).

In [11]:
print("Cola de pedidos pendientes (del mas antiguo al mas nuevo):")
for raw in r.lrange(COLA, 0, -1):
    p = json.loads(raw)
    print(f"   venta {p['venta_id']} | {p['sucursal']:<10} | {p['n_items']} items | ${p['total']}")

print("\nLLEN (cantidad pendiente):", r.llen(COLA))

prox = json.loads(r.lindex(COLA, 0))
print(f"Proximo a despachar (LINDEX 0): venta {prox['venta_id']} - {prox['sucursal']}")

Cola de pedidos pendientes (del mas antiguo al mas nuevo):
   venta 1050 | Centro     | 2 items | $62000
   venta 1051 | Centro     | 1 items | $9900
   venta 1052 | Caballito  | 4 items | $89000
   venta 1053 | Palermo    | 2 items | $47500
   venta 1054 | Palermo    | 2 items | $38900

LLEN (cantidad pendiente): 5
Proximo a despachar (LINDEX 0): venta 1050 - Centro


### Gestion: despachar, encolar y cancelar
`LPOP` despacha el mas antiguo, `RPUSH` encola una venta nueva y `RPOP` cancela la ultima ingresada.

In [12]:
# LPOP: despacha el mas antiguo (FIFO)
desp = despachar_pedido()
print(f"Despachado: venta {desp['venta_id']} ({desp['sucursal']}). Quedan {r.llen(COLA)}")

# RPUSH: una venta nueva entra al final
nuevo = encolar_pedido(nuevo_pedido(1055))
print(f"Encolada venta nueva {nuevo['venta_id']}. Quedan {r.llen(COLA)}")

# RPOP: cancela la ultima ingresada
cancelado = json.loads(r.rpop(COLA))
print(f"Cancelada (RPOP) la ultima ingresada: venta {cancelado['venta_id']}. Quedan {r.llen(COLA)}")

Despachado: venta 1050 (Centro). Quedan 4
Encolada venta nueva 1055. Quedan 5
Cancelada (RPOP) la ultima ingresada: venta 1055. Quedan 4


### Simulacion del flujo
Intercalamos ventas (`RPUSH`) y despachos (`LPOP`) mostrando como crece y se vacia la cola.

In [13]:
r.delete(COLA)
random.seed(99)

print("=== Simulacion del flujo de pedidos ===\n")
eventos = ["venta", "venta", "despacho", "venta", "despacho",
           "despacho", "venta", "venta", "despacho"]
vid = 2000
for ev in eventos:
    if ev == "venta":
        p = encolar_pedido(nuevo_pedido(vid))
        print(f"[VENTA]    venta {p['venta_id']} encolada ({p['sucursal']:<10}) -> pendientes: {r.llen(COLA)}")
        vid += 1
    else:
        p = despachar_pedido()
        if p:
            print(f"[DESPACHO] sale  venta {p['venta_id']} ({p['sucursal']:<10}) -> pendientes: {r.llen(COLA)}")
        else:
            print("[DESPACHO] cola vacia, nada para despachar")

print(f"\nPedidos que quedaron sin despachar: {r.llen(COLA)}")

=== Simulacion del flujo de pedidos ===

[VENTA]    venta 2000 encolada (Caballito ) -> pendientes: 1
[VENTA]    venta 2001 encolada (Caballito ) -> pendientes: 2
[DESPACHO] sale  venta 2000 (Caballito ) -> pendientes: 1
[VENTA]    venta 2002 encolada (Belgrano  ) -> pendientes: 2
[DESPACHO] sale  venta 2001 (Caballito ) -> pendientes: 1
[DESPACHO] sale  venta 2002 (Belgrano  ) -> pendientes: 0
[VENTA]    venta 2003 encolada (Centro    ) -> pendientes: 1
[VENTA]    venta 2004 encolada (Centro    ) -> pendientes: 2
[DESPACHO] sale  venta 2003 (Centro    ) -> pendientes: 1

Pedidos que quedaron sin despachar: 1


## 8. Datos con tiempo de vida (TTL) (4.1.3)
Tres claves que expiran solas: carrito (30 min), reserva de stock (10 min) y token de sesion (1 hora).

In [14]:
# Carrito de compras (hash): expira a los 30 min
r.delete("carrito:cliente:842")
r.hset("carrito:cliente:842", mapping={"1": 2, "12": 1})
r.expire("carrito:cliente:842", 1800)

# Reserva de stock: se libera a los 10 min
r.set("reserva:stock:1:3", 5, ex=600)

# Token de sesion: vence en 1 hora
r.set("sesion:token:abc123", 842, ex=3600)

print("3 claves creadas con TTL:")
print("  carrito:cliente:842   ->", r.ttl("carrito:cliente:842"), "s (30 min)")
print("  reserva:stock:1:3     ->", r.ttl("reserva:stock:1:3"), "s (10 min)")
print("  sesion:token:abc123   ->", r.ttl("sesion:token:abc123"), "s (1 hora)")

3 claves creadas con TTL:
  carrito:cliente:842   -> 1800 s (30 min)
  reserva:stock:1:3     -> 600 s (10 min)
  sesion:token:abc123   -> 3600 s (1 hora)


### Tiempo restante (`TTL` / `PTTL`)
`TTL` devuelve segundos; `PTTL`, milisegundos. `-1` = sin expiracion; `-2` = la clave ya no existe.

In [15]:
t_token = r.ttl("sesion:token:abc123")
t_res   = r.ttl("reserva:stock:1:3")

print(f"El token del usuario 842 sigue vigente por {t_token} segundos.")
print(f"La reserva del producto 1 en sucursal 3 se liberara en {t_res} segundos.")
print(f"PTTL del token (milisegundos): {r.pttl('sesion:token:abc123')}")

El token del usuario 842 sigue vigente por 3600 segundos.
La reserva del producto 1 en sucursal 3 se liberara en 600 segundos.
PTTL del token (milisegundos): 3599996


### Renovacion y expiracion automatica
Renovamos el TTL del carrito con `EXPIRE` y mostramos la expiracion real con una reserva de TTL corto (2 s).

In [16]:
import time

# Renovamos el TTL del carrito (cada accion del cliente lo reinicia)
r.expire("carrito:cliente:842", 1800)
print("El cliente agrego un producto -> TTL del carrito reiniciado a",
      r.ttl("carrito:cliente:842"), "s\n")

# Reserva con TTL corto para mostrar la expiracion
r.set("reserva:stock:9:2", 3, ex=2)
print("Reserva temporal creada (TTL 2s). Vigente ahora?:", r.exists("reserva:stock:9:2") == 1)

time.sleep(3)

if r.exists("reserva:stock:9:2") == 0:
    print("La reserva del producto 9 expiro automaticamente: las 3 unidades se liberan.")
print("TTL de la clave expirada:", r.ttl("reserva:stock:9:2"), "(-2 = ya no existe)")

El cliente agrego un producto -> TTL del carrito reiniciado a 1800 s

Reserva temporal creada (TTL 2s). Vigente ahora?: True


La reserva del producto 9 expiro automaticamente: las 3 unidades se liberan.
TTL de la clave expirada: -2 (-2 = ya no existe)
